# Validation by Comparison with FEniCS

This notebook presents how the validation suite compares SOFA with [FEniCS](https://fenicsproject.org), and walks through the code that does it on one case.

## Validation in a nutshell

A case is a boundary-value problem solved twice: once by SOFA, with the component under test, and once by FEniCS, an independent finite element library. Both solve it on the same mesh, so both give a displacement at the same nodes, and the two are compared node by node. Where the problem has an exact solution, SOFA is compared with it as well.

### Measuring the difference

The difference is measured by its relative RMS over the $N$ nodes of the mesh, where $\mathbf{u}_i$ is SOFA's displacement at node $i$ and $\mathbf{u}_i^\text{ref}$ the reference one, from FEniCS or the exact solution:

$$e_\text{rms} = \sqrt{\frac{\sum_{i=1}^{N} \|\mathbf{u}_i - \mathbf{u}_i^\text{ref}\|^2}{\sum_{i=1}^{N} \|\mathbf{u}_i^\text{ref}\|^2}}$$

A case passes when $e_\text{rms}$ against FEniCS is below the case's `tolerance`, and, if the case has an exact solution, $e_\text{rms}$ against it is below its `toleranceAnalytic`.

## Example: axial compression of a 2D beam

The rest of this notebook walks through one case, `validation/cases/LinearSmallStrainFEMForceField/2D/AxialCompression/`, building each piece of it in turn.

### The problem

The beam starts at the origin, with length $L$ along $x$ and width $W$ along $y$. Its right side carries a uniform compressive traction $p$. The left side is held only along $x$, so that the beam can contract freely across its width, and the stress is uniform, $\sigma_{xx} = -p$. The exact solution is

$$u_x = -\frac{p}{E}\,x, \qquad u_y = \frac{\nu p}{E}\,y$$

- `left` ($x = 0$): $u_x = 0$
- corner $(0, 0)$: $u_y = 0$
- `right` ($x = L$): $\mathbf{t} = (-p, 0)$
- `bottom` and `top`: $\mathbf{t} = 0$

### The case folder

A case is a folder, named after the problem and placed under its component and dimension:

```
AxialCompression/
├── case.json                 the input file, with one entry per element
├── sofa_scene.py             builds the problem in SOFA and solves it
├── fenics_scene.py           builds the same problem in FEniCS and solves it
├── reference_solution.json   FEniCS's solution, written by fenics_scene.py
└── analytic_solution.py      the exact solution, for the cases that have one
```

Each of these is introduced below, where the walkthrough reaches it.

### The input file

`case.json` holds one entry per element the case is run on, here `quad` and `tri`. The walkthrough uses `quad`. Its keys:

- `geometry` names the geometry and gives its size, $L$ and $W$;
- `element` is the kind of element the geometry is meshed with;
- `material` names the constitutive law and gives its parameters, $E$ and $\nu$;
- `forceField` is the SOFA component under test;
- `solvers` sets how SOFA solves the discrete system;
- `mesh` gives the number of grid points along each axis;
- `load` gives the traction $p$;
- `tolerance` and `toleranceAnalytic` are the thresholds on $e_\text{rms}$.

In [ ]:
import json
import pathlib
import sys

# Make the plugin's packages importable from validation/docs
sys.path.insert(0, "../..")

CASE_DIR = pathlib.Path("../cases/LinearSmallStrainFEMForceField/2D/AxialCompression")
case = json.loads((CASE_DIR / "case.json").read_text())["quad"]
case

### Solving with SOFA

The scene is built in three steps:

1. **The body.** `ElasticBeam` (`common/sofa/prefabs/grid.py`) meshes the beam with a regular grid of `quad` elements and adds the force field and the solvers.
2. **The boundary conditions.** `RegionClamp` (`common/sofa/controllers.py`) fixes $u_x$ on the `left` side, and a constraint fixes $u_y$ at the corner $(0, 0)$.
3. **The traction.** The traction $(-p, 0)$ is integrated over the edges of the `right` side into nodal loads.

In [ ]:
import numpy as np
import Sofa.Core
import Sofa.Simulation

from common.geometry import Beam2D
from common.sofa.controllers import RegionClamp
from common.sofa.prefabs import ElasticBeam

root = Sofa.Core.Node("root")
root.addObject("RequiredPlugin", pluginName=[
    "Sofa.Component.Constraint.Projective",
    "Sofa.Component.LinearSolver.Direct",
    "Sofa.Component.ODESolver.Backward",
    "Sofa.Component.SolidMechanics.FEM.Elastic",
    "Sofa.Component.StateContainer",
    "Sofa.Component.Topology.Container.Dynamic",
    "Sofa.Component.Topology.Container.Grid",
    "Sofa.Component.Topology.Mapping",
])
root.addObject("DefaultAnimationLoop")

# 1.
geometry = Beam2D(length=case["geometry"]["length"], width=case["geometry"]["width"])
configs = {"material": case["material"], "forceField": case["forceField"], "solvers": case["solvers"]}
beam = root.addChild(ElasticBeam(name="ElasticBeam", geometry=geometry, element=case["element"],
                                 grid_resolution=case["mesh"]["resolution"], configs=configs))
body = beam.mechanical_node()

# The grid creates its nodes only once initialised, and the regions are looked up on them
beam.Grid.grid.init()
rest_positions = beam.Grid.grid.position.array()[:, :2]
corner = sorted(set(geometry.region_indices("left", rest_positions))
                & set(geometry.region_indices("bottom", rest_positions)))
right_nodes = geometry.region_indices("right", rest_positions)

# 2.
body.addObject(RegionClamp(geometry=geometry, dofs=body.dofs, node=body, prescribe_displacement_on={"left": [1, 0]},
                           vec_type="Vec2d", name="clampCtrl"))
body.addObject("PartialFixedProjectiveConstraint", name="pinCorner", template="Vec2d", indices=corner,
               fixedDirections=[0, 1])

# 3.
with body.addChild("LoadedEdge") as loaded_edge:
    loaded_edge.addObject("EdgeSetTopologyContainer", name="edges",
                          edges=[[right_nodes[k], right_nodes[k + 1]] for k in range(len(right_nodes) - 1)])
    loaded_edge.addObject("NodalSourceDensity", name="loadDensity", template="Vec2d",
                          property=[[-case["load"]["magnitude"], 0.0]])
    loaded_edge.addObject("VectorSourceTerm", name="traction", template="Vec2d,Edge", sourceDensity="@loadDensity")
    loaded_edge.addObject("FEMSourceTermIntegrator", name="load", template="Vec2d,Edge", topology="@edges",
                          quadratureDegree=1, constantSources="@traction")

# Initialise the scene, then solve it: one step of the static solver
Sofa.Simulation.init(root)
Sofa.Simulation.animate(root, root.dt.value)

# SOFA's solution at the nodes: where it moved them, minus where they started
x = body.dofs.rest_position.array().copy()
u = body.dofs.position.array() - x

Steps 1 to 3 are the `build` function of the case's `sofa_scene.py`, and its `solve(case)` builds, solves and returns $(\mathbf{x}, \mathbf{u})$ as above. The file is also a SOFA scene, to inspect a case in the SOFA GUI:

```bash
runSofa -l SofaPython3 validation/cases/LinearSmallStrainFEMForceField/2D/AxialCompression/sofa_scene.py --argv quad
```

### The FEniCS reference

`fenics_scene.py` sets up the same problem in FEniCS, as a weak form, with the same material, loads and boundary conditions. Its mesh, built by `BeamMesh` (`common/fenics/prefabs/grid.py`), reproduces SOFA's: the same nodes, the same elements and, for `tri`, the same split of each grid cell into triangles. FEniCS numbers the nodes its own way, so the solution is put back in SOFA's node order before it is saved.

Running `fenics_scene.py` writes FEniCS's nodes and displacements to `reference_solution.json`, one entry per element. The file is committed with the case, so FEniCS is needed only to regenerate it, not to run the suite.

In [ ]:
reference = json.loads((CASE_DIR / "reference_solution.json").read_text())["quad"]["fenics"]
x_fenics, u_fenics = np.array(reference["x"]), np.array(reference["u"])

# The same nodes in the same order, so the solutions compare node by node
assert np.allclose(x, x_fenics, atol=1e-8)

### Comparing

$e_\text{rms}$ is computed by `RMSRelative` (`validation/study/metrics.py`). The exact solution is the `displacement(x, case)` function of the case's `analytic_solution.py`.

The exact displacement is linear in $x$ and $y$, and linear elements reproduce it exactly, so both differences are at round-off level. This is why this case's tolerances are $10^{-10}$.

In [ ]:
from validation.study.metrics import RMSRelative

sys.path.insert(0, str(CASE_DIR))
from analytic_solution import displacement

u_exact = displacement(x, case)
rms = RMSRelative()
print(f"e_rms against FEniCS:             {rms.measure(u, u_fenics):.2e} (tolerance {case['tolerance']['rms']:.0e})")
print(f"e_rms against the exact solution: {rms.measure(u, u_exact):.2e} (tolerance {case['toleranceAnalytic']['rms']:.0e})")

The plot draws the three solutions along two sides of the beam: $u_x$ along `bottom`, and $u_y$ along `right`.

In [ ]:
import matplotlib.pyplot as plt

on_bottom = np.isclose(x[:, 1], 0.0)
on_right = np.isclose(x[:, 0], geometry.length)

figure, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for ax, nodes, axis, name in ((axes[0], on_bottom, 0, "x"), (axes[1], on_right, 1, "y")):
    ax.plot(x[nodes, axis], u_exact[nodes, axis], "k--", lw=1, label="exact")
    ax.plot(x[nodes, axis], u_fenics[nodes, axis], "s", mfc="none", mec="tab:blue", ms=8, mew=1.5, label="FEniCS")
    ax.plot(x[nodes, axis], u[nodes, axis], "+", color="tab:red", ms=8, mew=1.5, label="SOFA")
    ax.set_xlabel(f"${name}$")
    ax.set_ylabel(f"$u_{name}$")
    ax.grid(alpha=0.3)
axes[0].set_title("$u_x$ along the bottom side")
axes[1].set_title("$u_y$ along the right side")
axes[0].legend()
plt.show()

## Putting it together

Everything the walkthrough did by hand is done by `ISFComparisonStudy` (`validation/study/study.py`), for every element of `case.json`:

- it loads FEniCS's solution from `reference_solution.json`, running `fenics_scene.py` first if the file has no entry for the element;
- it calls `solve(case)` of `sofa_scene.py`, and checks that SOFA's nodes are FEniCS's;
- it measures $e_\text{rms}$ against FEniCS and, if `analytic_solution.py` exists, against the exact solution, and decides whether the element passes;
- it draws a figure per element: a row each for SOFA, FEniCS and the exact solution, a column per component of $\mathbf{u}$.

`run` (`validation/run.py`) runs the study and saves its figures under `validation/results/`.

In [ ]:
from validation.run import run

study = run(CASE_DIR)
for element in study.results:
    print(f"{element}: e_rms = {study.results[element]['rms']:.2e} against FEniCS, "
          f"{study.analytic_results[element]['rms']:.2e} against the exact solution, "
          f"{'pass' if study.passed[element] else 'fail'}")

study.plots["quad"]

The same runs from the command line:

```bash
python validation/run.py validation/cases/LinearSmallStrainFEMForceField/2D/AxialCompression
python validation/run.py --all       # every case, then one summary line per case and element
python validation/run.py --all --fenics   # regenerate every reference_solution.json first
```

The cases of a component are presented in its own notebook, e.g. [LinearSmallStrainFEMForceField.ipynb](LinearSmallStrainFEMForceField.ipynb).